# 4. Data Modeling & Training (via mBERT) - Akkadian to English Translation 
## Aaron Dichoso & Luis Razon

This notebook details the steps performed for fine tuning MBart on the cleaned dataset used in the Deep Past Challenge for Translating Akkadian Text to English.
The competition can be accessed in this link: https://www.kaggle.com/competitions/deep-past-initiative-machine-translation/data

Run this notebook AFTER running "2. Preprocessing". This notebook does not depend on operations done in "3. ModelingTraining".

Most steps performed in this notebook, however, are the same as the ones done in "3. ModelingTraining". So please consult that notebook for details for some of the decisions in this notebook.



MBart (Multilingual BART) is a pretrained sequence-to-sequence transformer developed by Facebook AI, trained on 25 languages using a denoising autoencoder objective (Source: https://arxiv.org/pdf/2001.08210). It uses a standard encoder-decoder architecture where the encoder learns contextual representations of the source language and the decoder generates the target language autoregressively, with cross-attention connecting the two. Its multilingual pretraining means it has already learned general translation behaviors, like how to align source and target sequences, how to handle long-range dependencies, and how to generate fluent output.

We chose MBart because this pretrained knowledge transfers even to languages not included in its training. Rather than learning translation from scratch on our small dataset, we can just fine-tune MBart for Akkadian, giving it a significant head start over our custom transformer and LSTM models which were trained from scratch. 

This requires a different preprocessing stage, however. This is because MBart handles the tokenization by itself.

In [ ]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import (
    MBartForConditionalGeneration,
    MBart50TokenizerFast,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

For the MBart implementation, we do not use the tokenized data as it has its own built-in tokenizers.

In [ ]:
processed_complete_df = pd.read_csv("processed/cleaned_train_complete.csv")
processed_incomplete_df = pd.read_csv("processed/cleaned_train_incomplete.csv")

processed_complete_df.sample(5)

,oare_id,transliteration,translation
978,9cc6ae0d-4500-4ff9-b99b-fb1ddc4220b5,um-ma i-ku-pì-a-ma a-na PUZUR4.IŠTAR ù en-um-a...,From Ikūn-pīya to Puzur-Ištar and Ennam-Aššur:...
211,22139769-6254-4c9e-8c85-10e3cd7f095c,a-na šu-IŠTAR qí-bi-ma um-ma zu-ba ma-ta-ga-<g...,"Say to Šu-Ištar, thus Zuba, Mataga <big_gap>, ..."
1441,f0286b3b-ac61-46fb-b7a7-9afbb73abe91,um-ma šu-IŠTAR-ma ù la-qé-pu-um-ma a-na tám-kà...,From Šu-Ištar and Lā-qēp to the merchant: Iddi...
1498,f8df61cc-3eb3-4867-9bc1-515d1bad0a59,18 TÚG šu-ru-tim ù 10 (TÚG)ku-ta-ni 2 GÚ AN.NA...,"18 dark textiles and 10 textiles, 2 talents of..."
591,5d3b00b0-7e28-4125-b8c1-a80975b58ec0,um-ma en-um-a-šùr ú da-dí-a-ma a-na e-lá-ma qí...,"Thus Ennam-Aššur and Dadiya, say to Elamma: He..."


In [ ]:
akkadian_texts = processed_complete_df["transliteration"].tolist()
english_texts  = processed_complete_df["translation"].tolist()

print(f"Total pairs:      {len(processed_complete_df)}")
print(f"Sample Akkadian:  {akkadian_texts[0]}")
print(f"Sample English:   {english_texts[0]}")

# Check sequence lengths to set max_len appropriately
akk_lens = processed_complete_df["transliteration"].str.split().str.len()
eng_lens  = processed_complete_df["translation"].str.split().str.len()
print(f"\nAkkadian lengths — mean: {akk_lens.mean():.1f} | max: {akk_lens.max()}")
print(f"English lengths  — mean: {eng_lens.mean():.1f}  | max: {eng_lens.max()}")

Total pairs:      1536
Sample Akkadian:  KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠIB šu-(d)EN.LÍL DUMU ma-nu-ki-a-šur KIŠIB MAN-a-šur DUMU a-ta-a 1/3 ma-na 2 GÍN KÙ.BABBAR SIG5 i-ṣé-er PUZUR4-a-šur DUMU a-ta-a-lá-hu-um i-šu iš-tù ha-muš-tim ša ì-lí-dan ITU.KAM ša ke-na-tim li-mu-um e-na-sú-in a-na ITU 14 ha-am-ša-tim i-ša-qal šu-ma lá iš-qú-ul 1/2 GÍN.TA a-na 1 ma-na-im i-na ITU.1.KAM ṣí-ib-tám ú-ṣa-áb
Sample English:   Seal of Mannum-balum-Aššur son of Ṣilli-Adad, seal of Šu-Illil son of Mannum-kī-Aššur, seal of Puzur-Aššur son of Ataya. Puzur-Aššur son of Ataya owes 22 shekels of good silver to Ali-ahum. Reckoned from the week of Ilī-dan, month of Ša-kēnātim, in the eponymy of Enna-Suen, he will pay in 14 weeks. If he has not paid in time, he will add interest at the rate 1/2 shekel per mina per month.

Akkadian lengths — mean: 55.7 | max: 158
English lengths  — mean: 91.3  | max: 748


In [ ]:
MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
model     = MBartForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)

Loading weights: 100%|██████████| 516/516 [00:00<00:00, 2846.91it/s]


The first step is to extend MBart's tokenizer with the Akkadian transliterations.

In [ ]:
import utils.bpe as bpe

BPE = bpe.BytePairEncoder()
BPE.load("processed/akkonly.json")

#NOTE: AI was used to help understand the flow in setting up MBart Model for training and testing and for general debugging. 
#The following code, however, was written and verified by the authors. 

# Extract learned BPE subwords from your trained BytePairEncoder
def get_bpe_subwords(bpe_encoder, top_n=200):
    subwords = []
    for token, freq in bpe_encoder.tokens.most_common():
        clean = token.replace('_', '').strip()
        if (len(clean) > 1
            and not (clean.startswith('<') and clean.endswith('>'))
            and clean not in subwords
        ):
            subwords.append(clean)
        if len(subwords) >= top_n:
            break
    return subwords

# Akkadian determinative tokens used in transliteration
AKK_SPECIAL_TOKENS = [
    "<god>", "<star>", "<place>", "<person>", "<building>",
    "<city>", "<land>", "<female>", "<male>", "<wood>",
    "<textile>", "<tablet>", "<river>", "<bird>", "<stone>",
    "<hide>", "<plant>",
    "<sos>", "<eos>", "<pad>", "<unk>",  # base special tokens
]

# Subwords obtained from BPE
AKK_SUBWORDS = get_bpe_subwords(BPE, top_n=200)
print(f"Extracted {len(AKK_SUBWORDS)} subwords from BPE model")
print(f"Top 50: {AKK_SUBWORDS[:50]}")

# Add custom language token for Akkadian
AKKADIAN_LANG_TOKEN = "akk_XX"

Model loaded from processed/akkonly.json
Extracted 200 subwords from BPE model
Top 50: ['ma', 'na', 'BA', 'im', 'ša', 'ni', 'šu', 'um', 'lá', 'KÙ', 'KÙ.', 'ku', 'am', 'BAB', 'BABBA', 'KÙ.BABBA', 'KÙ.BABBAR', 'tí', 'ur', 'ta', 'DU', 'nu', 'kà', 'dí', 'li', 'GÍ', 'šur', 'GÍN', 'ba', 'DUM', 'DUMU', 'tim', 'lu', 'ší', 'iš', 'GI', 'ra', 'ha', 'ki', 'bi', 'ri', 'mì', 'bu', 'tù', '1/', 'IŠ', 'ru', 'šù', 'šùr', 'qí']


In [ ]:
# Register all new tokens
existing_special_tokens = tokenizer.special_tokens_map.get("additional_special_tokens", [])
num_added = tokenizer.add_special_tokens({
    "additional_special_tokens": (
        existing_special_tokens
        + AKK_SPECIAL_TOKENS
        + [AKKADIAN_LANG_TOKEN]
    )
})
tokenizer.add_tokens(AKK_SUBWORDS)

# Resize model embeddings to account for new tokens
model.resize_token_embeddings(len(tokenizer))

# Set Akkadian as the source language
tokenizer.src_lang = AKKADIAN_LANG_TOKEN
tokenizer.tgt_lang = "en_XX"

# Register the new lang token ID so model.generate() can use forced_bos correctly
tokenizer.lang_code_to_id[AKKADIAN_LANG_TOKEN] = tokenizer.convert_tokens_to_ids(AKKADIAN_LANG_TOKEN)

print(f"Added {num_added} special tokens + {len(AKK_SUBWORDS)} subwords")
print(f"New vocab size: {len(tokenizer)}")
print(f"Akkadian lang token ID: {tokenizer.lang_code_to_id[AKKADIAN_LANG_TOKEN]}")

Added 20 special tokens + 200 subwords
New vocab size: 250162
Akkadian lang token ID: 250073


In [ ]:
class AkkadianDataset(Dataset):
    def __init__(self, src_texts, tgt_texts, tokenizer, max_len=128):
        self.src_texts = src_texts
        self.tgt_texts = tgt_texts
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.src_texts)

    def __getitem__(self, idx):
        model_inputs = self.tokenizer(
            self.src_texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        with self.tokenizer.as_target_tokenizer():
            labels = self.tokenizer(
                self.tgt_texts[idx],
                max_length=self.max_len,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )

        label_ids = labels["input_ids"].squeeze()
        label_ids[label_ids == tokenizer.pad_token_id] = -100

        return {
            "input_ids":      model_inputs["input_ids"].squeeze(),
            "attention_mask": model_inputs["attention_mask"].squeeze(),
            "labels":         label_ids
        }

In [ ]:
MAX_LEN = 748

dataset   = AkkadianDataset(akkadian_texts, english_texts, tokenizer, max_len=MAX_LEN)
total     = len(dataset)
train_len = int(total * 0.8)
val_len   = total - train_len

train_set, val_set = random_split(
    dataset,
    [train_len, val_len],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_set, batch_size=8,  shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=8,  shuffle=False, num_workers=2, pin_memory=True)

print(f"\nTrain: {train_len} | Val: {val_len}")


Train: 1228 | Val: 308


In [ ]:
def train_mbart(model, train_loader, val_loader, epochs=30, save_path="checkpoints/mbart"):
    os.makedirs(save_path, exist_ok=True)

    optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=len(train_loader) * 3,
        num_training_steps=len(train_loader) * epochs
    )

    best_val_loss = float("inf")

    #Training Loop
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        torch.cuda.empty_cache()

        for batch in train_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            optimizer.zero_grad()
            output = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            output.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            train_loss += output.loss.item()

        # ── Validate ──
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids      = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels         = batch["labels"].to(device)

                output = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                val_loss += output.loss.item()

        avg_train = train_loss / len(train_loader)
        avg_val   = val_loss   / len(val_loader)

        print(f"Epoch {epoch+1:03d} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            model.save_pretrained(f"save_path/E{epoch}")
            tokenizer.save_pretrained(f"save_path/E{epoch}")
            print(f"Saved checkpoint (val loss: {avg_val:.4f})")

In [ ]:
def translate(text, max_len=128):
    model.eval()
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_len
    ).to(device)

    with torch.no_grad():
        translated = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.lang_code_to_id["en_XX"],
            max_length=max_len,
            num_beams=4,
            length_penalty=0.6,   # penalise long outputs
            early_stopping=True
        )

    return tokenizer.batch_decode(translated, skip_special_tokens=True)[0]

In [ ]:
def load_checkpoint(save_path="checkpoints/mbart"):
    model     = MBartForConditionalGeneration.from_pretrained(save_path).to(device)
    tokenizer = MBart50TokenizerFast.from_pretrained(save_path)
    tokenizer.src_lang = AKKADIAN_LANG_TOKEN
    tokenizer.tgt_lang = "en_XX"
    return model, tokenizer

In [ ]:
train_mbart(model, train_loader, val_loader, epochs=30)

# Test a translation
sample = akkadian_texts[0]
print(f"\nSource:    {sample}")
print(f"Expected:  {english_texts[0]}")
print(f"Predicted: {translate(sample)}")

Note: no notebook outputs show up in here as we trained MBart on a cloud server using a separate python file containing the same flow seen above.

MBart Training Ends Here.